# Notebook 05 — Analyse des KPI

## Objectif

L'objectif de ce notebook est de calculer les principaux indicateurs
de performance (KPI) nécessaires au suivi des patients et des séances
de rééducation.

Les KPI seront calculés à plusieurs niveaux :

- niveau global ;
- niveau patient ;
- niveau exercice ;
- niveau jeu ;
- niveau niveau de difficulté.

Ces indicateurs serviront ensuite à construire le dashboard destiné
aux thérapeutes.

Les principaux indicateurs étudiés sont :

- nombre de patients ;
- nombre de sessions ;
- score moyen ;
- taux de réussite moyen ;
- durée moyenne des sessions ;
- nombre moyen de répétitions ;
- progression moyenne ;
- nombre de sessions avec amélioration ;
- nombre de sessions avec diminution.

In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

In [2]:
PROCESSED_PATH = "../data/processed/"
session_analysis = pd.read_csv(
    PROCESSED_PATH + "session_analysis.csv"
)

In [3]:
session_analysis["session_date"] = pd.to_datetime(
    session_analysis["session_date"]
)

session_analysis["date_of_birth"] = pd.to_datetime(
    session_analysis["date_of_birth"]
)

In [4]:
print("Dimensions :", session_analysis.shape)

print("\nDate session :")
print(session_analysis["session_date"].dtype)

print("\nDate naissance :")
print(session_analysis["date_of_birth"].dtype)

Dimensions : (500, 36)

Date session :
datetime64[ns]

Date naissance :
datetime64[ns]


In [5]:
total_patients = session_analysis["patient_id"].nunique()

print("Nombre total de patients :", total_patients)

Nombre total de patients : 20


In [6]:
total_sessions = session_analysis["id_session"].nunique()

print("Nombre total de sessions :", total_sessions)

Nombre total de sessions : 500


In [7]:
total_exercises = session_analysis["exercise_id"].nunique()

print("Nombre total d'exercices :", total_exercises)

Nombre total d'exercices : 8


In [8]:
total_games = session_analysis["game_id"].nunique()

print("Nombre total de jeux :", total_games)

Nombre total de jeux : 3


In [9]:
average_score = session_analysis["score"].mean()

print("Score moyen :", round(average_score, 2))

Score moyen : 56.78


In [10]:
average_success_rate = session_analysis["success_rate"].mean()

print(
    "Taux de réussite moyen :",
    round(average_success_rate, 2),
    "%"
)

Taux de réussite moyen : 57.34 %


In [11]:
average_duration = session_analysis["duration"].mean()

print(
    "Durée moyenne :",
    round(average_duration, 2),
    "minutes"
)

Durée moyenne : 7.84 minutes


In [12]:
average_repetitions = session_analysis["repetitions"].mean()

print(
    "Répétitions moyennes :",
    round(average_repetitions, 2)
)

Répétitions moyennes : 14.16


In [13]:
average_progression = session_analysis["progression"].mean()

print(
    "Progression moyenne :",
    round(average_progression, 2),
    "%"
)

Progression moyenne : 3.0 %


In [14]:
sessions_with_progression = (
    session_analysis["progression"]
    .notna()
    .sum()
)

print(
    "Sessions avec progression calculable :",
    sessions_with_progression
)

Sessions avec progression calculable : 457


In [15]:
improved_sessions = (
    session_analysis["progression"] > 0
).sum()

print(
    "Sessions avec amélioration :",
    improved_sessions
)

Sessions avec amélioration : 260


In [16]:
declined_sessions = (
    session_analysis["progression"] < 0
).sum()

print(
    "Sessions avec diminution :",
    declined_sessions
)

Sessions avec diminution : 196


In [17]:
improvement_rate = (
    improved_sessions
    / sessions_with_progression
    * 100
)

print(
    "Taux de sessions avec amélioration :",
    round(improvement_rate, 2),
    "%"
)

Taux de sessions avec amélioration : 56.89 %


In [18]:
decline_rate = (
    declined_sessions
    / sessions_with_progression
    * 100
)

print(
    "Taux de sessions avec diminution :",
    round(decline_rate, 2),
    "%"
)

Taux de sessions avec diminution : 42.89 %


In [19]:
global_kpis = pd.DataFrame({
    "KPI": [
        "Patients",
        "Sessions",
        "Exercices",
        "Jeux",
        "Score moyen",
        "Taux de réussite moyen",
        "Durée moyenne",
        "Répétitions moyennes",
        "Progression moyenne",
        "Sessions avec amélioration",
        "Sessions avec diminution"
    ],
    "Valeur": [
        total_patients,
        total_sessions,
        total_exercises,
        total_games,
        average_score,
        average_success_rate,
        average_duration,
        average_repetitions,
        average_progression,
        improved_sessions,
        declined_sessions
    ]
})

global_kpis["Valeur"] = global_kpis["Valeur"].round(2)

global_kpis

,KPI,Valeur
0,Patients,20.00
1,Sessions,500.00
2,Exercices,8.00
3,Jeux,3.00
4,Score moyen,56.78
5,Taux de réussite moyen,57.34
6,Durée moyenne,7.84
7,Répétitions moyennes,14.16
8,Progression moyenne,3.00
9,Sessions avec amélioration,260.00


In [20]:
patient_kpis = (
    session_analysis
    .groupby(
        ["patient_id", "first_name", "last_name"]
    )
    .agg(
        sessions=("id_session", "nunique"),
        mean_score=("score", "mean"),
        mean_success_rate=("success_rate", "mean"),
        mean_duration=("duration", "mean"),
        mean_repetitions=("repetitions", "mean"),
        mean_progression=("progression", "mean")
    )
    .reset_index()
)

In [21]:
numeric_columns = [
    "mean_score",
    "mean_success_rate",
    "mean_duration",
    "mean_repetitions",
    "mean_progression"
]

patient_kpis[numeric_columns] = (
    patient_kpis[numeric_columns].round(2)
)

patient_kpis

,patient_id,first_name,last_name,sessions,mean_score,mean_success_rate,mean_duration,mean_repetitions,mean_progression
0,1,Yasmine,Bouzid,26,60.97,60.91,6.62,10.50,3.24
1,2,Amira,Jebali,24,54.94,56.56,7.17,12.33,1.34
2,3,Fatma,Tlili,32,56.43,59.27,9.22,17.84,1.79
3,4,Malek,Hamdi,26,55.06,55.61,8.73,17.04,1.84
4,5,Zied,Gharbi,42,59.17,60.23,7.69,13.57,4.32
5,6,Eya,Haddad,66,59.53,59.47,7.73,14.89,3.06
6,7,Yassine,Sassi,31,57.83,58.08,8.03,13.61,3.48
7,8,Hiba,Khelifi,13,64.07,64.32,6.46,11.69,2.63
8,9,Adam,Mejri,25,55.15,55.92,6.08,9.40,2.56
9,10,Maya,Trabelsi,30,61.27,59.21,6.60,10.67,3.00


In [22]:
print("Nombre de lignes :", len(patient_kpis))
print(
    "Nombre de patients uniques :",
    patient_kpis["patient_id"].nunique()
)

Nombre de lignes : 20
Nombre de patients uniques : 20


In [23]:
patient_most_sessions = (
    patient_kpis
    .sort_values("sessions", ascending=False)
    .head(5)
)

patient_most_sessions

,patient_id,first_name,last_name,sessions,mean_score,mean_success_rate,mean_duration,mean_repetitions,mean_progression
5,6,Eya,Haddad,66,59.53,59.47,7.73,14.89,3.06
4,5,Zied,Gharbi,42,59.17,60.23,7.69,13.57,4.32
2,3,Fatma,Tlili,32,56.43,59.27,9.22,17.84,1.79
6,7,Yassine,Sassi,31,57.83,58.08,8.03,13.61,3.48
9,10,Maya,Trabelsi,30,61.27,59.21,6.60,10.67,3.00


In [24]:
top_score_patients = (
    patient_kpis
    .sort_values("mean_score", ascending=False)
    .head(5)
)

top_score_patients

,patient_id,first_name,last_name,sessions,mean_score,mean_success_rate,mean_duration,mean_repetitions,mean_progression
7,8,Hiba,Khelifi,13,64.07,64.32,6.46,11.69,2.63
9,10,Maya,Trabelsi,30,61.27,59.21,6.60,10.67,3.00
0,1,Yasmine,Bouzid,26,60.97,60.91,6.62,10.50,3.24
18,19,Mohamed,Slimani,12,60.83,62.30,7.17,12.50,4.93
5,6,Eya,Haddad,66,59.53,59.47,7.73,14.89,3.06


In [25]:
lowest_score_patients = (
    patient_kpis
    .sort_values("mean_score", ascending=True)
    .head(5)
)

lowest_score_patients

,patient_id,first_name,last_name,sessions,mean_score,mean_success_rate,mean_duration,mean_repetitions,mean_progression
11,12,Rim,Ayari,9,45.49,46.46,8.11,14.22,0.49
19,20,Omar,Dridi,25,48.31,49.78,8.00,14.80,0.48
16,17,Skander,Karoui,16,49.43,51.82,8.19,14.50,2.38
17,18,Lina,Chaouch,30,50.45,51.63,8.17,15.07,8.83
1,2,Amira,Jebali,24,54.94,56.56,7.17,12.33,1.34


In [26]:
exercise_kpis = (
    session_analysis
    .groupby(
        ["exercise_id", "name"]
    )
    .agg(
        sessions=("id_session", "nunique"),
        mean_score=("score", "mean"),
        mean_success_rate=("success_rate", "mean"),
        mean_duration=("duration", "mean"),
        mean_repetitions=("repetitions", "mean"),
        mean_progression=("progression", "mean")
    )
    .reset_index()
)

In [27]:
numeric_columns = [
    "mean_score",
    "mean_success_rate",
    "mean_duration",
    "mean_repetitions",
    "mean_progression"
]

exercise_kpis[numeric_columns] = (
    exercise_kpis[numeric_columns].round(2)
)

exercise_kpis

,exercise_id,name,sessions,mean_score,mean_success_rate,mean_duration,mean_repetitions,mean_progression
0,1,Forward Arm Raise,57,51.75,52.49,7.84,14.14,2.36
1,2,Lateral Arm Raise,30,60.43,61.29,8.20,14.50,3.23
2,3,Overhead Reach,17,50.45,53.78,7.88,14.94,1.35
3,4,Seated Knee Extension,56,59.24,59.92,7.38,13.14,3.79
4,5,Mini Squat,62,56.38,56.65,7.52,13.85,3.14
5,6,Step Up,79,58.37,58.65,7.73,13.85,2.16
6,7,Wrist Flexion and Extension,92,58.37,57.71,8.20,14.57,2.11
7,8,Forearm Rotation,107,55.84,57.15,7.94,14.56,4.45


In [28]:
game_kpis = (
    session_analysis
    .groupby(
        ["game_id", "name_game"]
    )
    .agg(
        sessions=("id_session", "nunique"),
        mean_score=("score", "mean"),
        mean_success_rate=("success_rate", "mean"),
        mean_duration=("duration", "mean"),
        mean_repetitions=("repetitions", "mean"),
        mean_progression=("progression", "mean")
    )
    .reset_index()
)

In [29]:
numeric_columns = [
    "mean_score",
    "mean_success_rate",
    "mean_duration",
    "mean_repetitions",
    "mean_progression"
]

game_kpis[numeric_columns] = (
    game_kpis[numeric_columns].round(2)
)

game_kpis

,game_id,name_game,sessions,mean_score,mean_success_rate,mean_duration,mean_repetitions,mean_progression
0,1,Bubble Reach,104,54.04,55.24,7.95,14.38,2.46
1,2,Step Quest,197,57.99,58.38,7.56,13.65,2.93
2,3,Wrist Wizard,199,57.01,57.41,8.06,14.56,3.36


In [30]:
level_kpis = (
    session_analysis
    .groupby("level_number")
    .agg(
        sessions=("id_session", "nunique"),
        mean_score=("score", "mean"),
        mean_success_rate=("success_rate", "mean"),
        mean_duration=("duration", "mean"),
        mean_repetitions=("repetitions", "mean"),
        mean_progression=("progression", "mean")
    )
    .reset_index()
)

In [31]:
numeric_columns = [
    "mean_score",
    "mean_success_rate",
    "mean_duration",
    "mean_repetitions",
    "mean_progression"
]

level_kpis[numeric_columns] = (
    level_kpis[numeric_columns].round(2)
)

level_kpis

,level_number,sessions,mean_score,mean_success_rate,mean_duration,mean_repetitions,mean_progression
0,1,65,51.98,53.45,8.49,16.11,1.72
1,2,248,56.05,56.82,7.92,14.29,3.52
2,3,187,59.41,59.38,7.52,13.32,2.76


In [32]:
progression_by_patient = (
    session_analysis[
        session_analysis["progression"].notna()
    ]
    .groupby(
        ["patient_id", "first_name", "last_name"]
    )
    .agg(
        sessions_with_progression=("progression", "count"),
        mean_progression=("progression", "mean"),
        max_progression=("progression", "max"),
        min_progression=("progression", "min")
    )
    .reset_index()
)

In [33]:
progression_columns = [
    "mean_progression",
    "max_progression",
    "min_progression"
]

progression_by_patient[progression_columns] = (
    progression_by_patient[progression_columns].round(2)
)

progression_by_patient

,patient_id,first_name,last_name,sessions_with_progression,mean_progression,max_progression,min_progression
0,1,Yasmine,Bouzid,24,3.24,21.52,-14.53
1,2,Amira,Jebali,22,1.34,28.46,-23.37
2,3,Fatma,Tlili,29,1.79,32.78,-25.09
3,4,Malek,Hamdi,24,1.84,37.40,-15.31
4,5,Zied,Gharbi,39,4.32,29.74,-18.71
5,6,Eya,Haddad,62,3.06,30.53,-20.93
6,7,Yassine,Sassi,29,3.48,27.73,-14.90
7,8,Hiba,Khelifi,12,2.63,14.08,-9.06
8,9,Adam,Mejri,23,2.56,17.69,-10.73
9,10,Maya,Trabelsi,28,3.00,28.02,-16.71


In [34]:
improvements_by_patient = (
    session_analysis[
        session_analysis["progression"] > 0
    ]
    .groupby(
        ["patient_id", "first_name", "last_name"]
    )
    .size()
    .reset_index(name="improved_sessions")
)

improvements_by_patient

,patient_id,first_name,last_name,improved_sessions
0,1,Yasmine,Bouzid,14
1,2,Amira,Jebali,11
2,3,Fatma,Tlili,16
3,4,Malek,Hamdi,11
4,5,Zied,Gharbi,26
5,6,Eya,Haddad,36
6,7,Yassine,Sassi,16
7,8,Hiba,Khelifi,8
8,9,Adam,Mejri,12
9,10,Maya,Trabelsi,18


In [35]:
declines_by_patient = (
    session_analysis[
        session_analysis["progression"] < 0
    ]
    .groupby(
        ["patient_id", "first_name", "last_name"]
    )
    .size()
    .reset_index(name="declined_sessions")
)

declines_by_patient

,patient_id,first_name,last_name,declined_sessions
0,1,Yasmine,Bouzid,10
1,2,Amira,Jebali,11
2,3,Fatma,Tlili,13
3,4,Malek,Hamdi,13
4,5,Zied,Gharbi,13
5,6,Eya,Haddad,26
6,7,Yassine,Sassi,13
7,8,Hiba,Khelifi,4
8,9,Adam,Mejri,10
9,10,Maya,Trabelsi,10


In [36]:
patient_dashboard_kpis = patient_kpis.merge(
    progression_by_patient[
        [
            "patient_id",
            "sessions_with_progression",
            "max_progression",
            "min_progression"
        ]
    ],
    on="patient_id",
    how="left"
)

In [37]:
patient_dashboard_kpis = patient_dashboard_kpis.merge(
    improvements_by_patient[
        [
            "patient_id",
            "improved_sessions"
        ]
    ],
    on="patient_id",
    how="left"
)

patient_dashboard_kpis = patient_dashboard_kpis.merge(
    declines_by_patient[
        [
            "patient_id",
            "declined_sessions"
        ]
    ],
    on="patient_id",
    how="left"
)

In [38]:
count_columns = [
    "sessions_with_progression",
    "improved_sessions",
    "declined_sessions"
]

patient_dashboard_kpis[count_columns] = (
    patient_dashboard_kpis[count_columns]
    .fillna(0)
    .astype(int)
)

patient_dashboard_kpis

,patient_id,first_name,last_name,sessions,mean_score,mean_success_rate,mean_duration,mean_repetitions,mean_progression,sessions_with_progression,max_progression,min_progression,improved_sessions,declined_sessions
0,1,Yasmine,Bouzid,26,60.97,60.91,6.62,10.50,3.24,24,21.52,-14.53,14,10
1,2,Amira,Jebali,24,54.94,56.56,7.17,12.33,1.34,22,28.46,-23.37,11,11
2,3,Fatma,Tlili,32,56.43,59.27,9.22,17.84,1.79,29,32.78,-25.09,16,13
3,4,Malek,Hamdi,26,55.06,55.61,8.73,17.04,1.84,24,37.40,-15.31,11,13
4,5,Zied,Gharbi,42,59.17,60.23,7.69,13.57,4.32,39,29.74,-18.71,26,13
5,6,Eya,Haddad,66,59.53,59.47,7.73,14.89,3.06,62,30.53,-20.93,36,26
6,7,Yassine,Sassi,31,57.83,58.08,8.03,13.61,3.48,29,27.73,-14.90,16,13
7,8,Hiba,Khelifi,13,64.07,64.32,6.46,11.69,2.63,12,14.08,-9.06,8,4
8,9,Adam,Mejri,25,55.15,55.92,6.08,9.40,2.56,23,17.69,-10.73,12,10
9,10,Maya,Trabelsi,30,61.27,59.21,6.60,10.67,3.00,28,28.02,-16.71,18,10


In [39]:
patient_dashboard_kpis["improvement_rate"] = (
    patient_dashboard_kpis["improved_sessions"]
    / patient_dashboard_kpis["sessions_with_progression"]
    * 100
)

In [40]:
patient_dashboard_kpis["improvement_rate"] = (
    patient_dashboard_kpis["improvement_rate"]
    .replace([np.inf, -np.inf], np.nan)
    .round(2)
)

patient_dashboard_kpis

,patient_id,first_name,last_name,sessions,mean_score,mean_success_rate,mean_duration,mean_repetitions,mean_progression,sessions_with_progression,max_progression,min_progression,improved_sessions,declined_sessions,improvement_rate
0,1,Yasmine,Bouzid,26,60.97,60.91,6.62,10.50,3.24,24,21.52,-14.53,14,10,58.33
1,2,Amira,Jebali,24,54.94,56.56,7.17,12.33,1.34,22,28.46,-23.37,11,11,50.00
2,3,Fatma,Tlili,32,56.43,59.27,9.22,17.84,1.79,29,32.78,-25.09,16,13,55.17
3,4,Malek,Hamdi,26,55.06,55.61,8.73,17.04,1.84,24,37.40,-15.31,11,13,45.83
4,5,Zied,Gharbi,42,59.17,60.23,7.69,13.57,4.32,39,29.74,-18.71,26,13,66.67
5,6,Eya,Haddad,66,59.53,59.47,7.73,14.89,3.06,62,30.53,-20.93,36,26,58.06
6,7,Yassine,Sassi,31,57.83,58.08,8.03,13.61,3.48,29,27.73,-14.90,16,13,55.17
7,8,Hiba,Khelifi,13,64.07,64.32,6.46,11.69,2.63,12,14.08,-9.06,8,4,66.67
8,9,Adam,Mejri,25,55.15,55.92,6.08,9.40,2.56,23,17.69,-10.73,12,10,52.17
9,10,Maya,Trabelsi,30,61.27,59.21,6.60,10.67,3.00,28,28.02,-16.71,18,10,64.29


In [41]:
print(
    patient_dashboard_kpis[
        [
            "mean_score",
            "mean_success_rate",
            "mean_progression",
            "improvement_rate"
        ]
    ].describe()
)

       mean_score  mean_success_rate  mean_progression  improvement_rate
count   20.000000          20.000000         20.000000          20.00000
mean    56.285000          56.909000          2.833500          56.28200
std      4.733777           4.371615          1.992434          10.83472
min     45.490000          46.460000          0.370000          37.50000
25%     55.030000          55.245000          1.677500          49.03750
50%     56.820000          58.150000          2.595000          56.61500
75%     59.260000          59.320000          3.300000          64.29000
max     64.070000          64.320000          8.830000          81.82000


In [42]:
recent_sessions = (
    session_analysis
    .sort_values("session_date", ascending=False)
    .head(10)
)

recent_sessions[
    [
        "id_session",
        "patient_id",
        "first_name",
        "last_name",
        "session_date",
        "name",
        "name_game",
        "level_number",
        "score",
        "success_rate",
        "progression"
    ]
]

,id_session,patient_id,first_name,last_name,session_date,name,name_game,level_number,score,success_rate,progression
475,500,16,Rayan,Mansouri,2026-09-22 16:30:42,Mini Squat,Step Quest,3,46.0,51.7,-14.50
381,499,5,Zied,Gharbi,2026-09-21 15:40:59,Forward Arm Raise,Bubble Reach,3,59.4,63.5,-2.14
264,498,6,Eya,Haddad,2026-09-19 11:45:07,Forward Arm Raise,Bubble Reach,3,61.6,70.6,3.53
288,497,5,Zied,Gharbi,2026-09-19 11:00:59,Mini Squat,Step Quest,3,73.9,69.6,0.27
171,496,8,Hiba,Khelifi,2026-09-19 10:45:56,Forearm Rotation,Wrist Wizard,2,72.5,65.3,3.57
421,495,12,Rim,Ayari,2026-09-19 10:45:39,Forward Arm Raise,Bubble Reach,1,48.0,45.8,17.65
202,494,6,Eya,Haddad,2026-09-19 10:45:27,Step Up,Step Quest,3,76.0,80.2,6.15
329,493,6,Eya,Haddad,2026-09-19 10:15:43,Mini Squat,Step Quest,2,71.6,68.2,25.61
403,492,5,Zied,Gharbi,2026-09-19 10:00:58,Lateral Arm Raise,Bubble Reach,3,73.0,71.5,2.24
227,491,6,Eya,Haddad,2026-09-19 09:30:52,Lateral Arm Raise,Bubble Reach,3,70.7,72.8,-1.94


In [43]:
daily_kpis = (
    session_analysis
    .groupby(
        session_analysis["session_date"].dt.date
    )
    .agg(
        sessions=("id_session", "nunique"),
        mean_score=("score", "mean"),
        mean_success_rate=("success_rate", "mean"),
        mean_duration=("duration", "mean"),
        mean_repetitions=("repetitions", "mean")
    )
    .reset_index()
)

In [44]:
daily_kpis[
    [
        "mean_score",
        "mean_success_rate",
        "mean_duration",
        "mean_repetitions"
    ]
] = daily_kpis[
    [
        "mean_score",
        "mean_success_rate",
        "mean_duration",
        "mean_repetitions"
    ]
].round(2)

daily_kpis.head(10)

,session_date,sessions,mean_score,mean_success_rate,mean_duration,mean_repetitions
0,2026-04-30,1,60.6,60.2,4.0,9.0
1,2026-05-04,1,52.5,52.7,8.0,9.0
2,2026-05-05,1,47.1,52.7,6.0,10.0
3,2026-05-07,1,63.8,68.7,5.0,10.0
4,2026-05-08,1,48.7,61.3,7.0,13.0
5,2026-05-12,2,58.9,60.7,5.0,6.5
6,2026-05-16,1,62.7,70.3,9.0,11.0
7,2026-05-18,1,52.9,55.8,6.0,9.0
8,2026-05-21,1,61.0,60.4,7.0,14.0
9,2026-05-22,1,50.6,54.7,8.0,13.0


In [45]:
print("===== VERIFICATION DES KPI =====")

print("Patients :", total_patients)
print("Sessions :", total_sessions)
print("Exercices :", total_exercises)
print("Jeux :", total_games)

print("\nPatient KPI :", patient_kpis.shape)
print("Exercise KPI :", exercise_kpis.shape)
print("Game KPI :", game_kpis.shape)
print("Level KPI :", level_kpis.shape)
print(
    "Patient Dashboard KPI :",
    patient_dashboard_kpis.shape
)

===== VERIFICATION DES KPI =====
Patients : 20
Sessions : 500
Exercices : 8
Jeux : 3

Patient KPI : (20, 9)
Exercise KPI : (8, 8)
Game KPI : (3, 8)
Level KPI : (3, 7)
Patient Dashboard KPI : (20, 15)


In [46]:
patient_kpis.to_csv(
    PROCESSED_PATH + "patient_kpis.csv",
    index=False
)

exercise_kpis.to_csv(
    PROCESSED_PATH + "exercise_kpis.csv",
    index=False
)

game_kpis.to_csv(
    PROCESSED_PATH + "game_kpis.csv",
    index=False
)

level_kpis.to_csv(
    PROCESSED_PATH + "level_kpis.csv",
    index=False
)

patient_dashboard_kpis.to_csv(
    PROCESSED_PATH + "patient_dashboard_kpis.csv",
    index=False
)

daily_kpis.to_csv(
    PROCESSED_PATH + "daily_kpis.csv",
    index=False
)

global_kpis.to_csv(
    PROCESSED_PATH + "global_kpis.csv",
    index=False
)

print("Tous les fichiers KPI ont été sauvegardés.")

Tous les fichiers KPI ont été sauvegardés.


In [ ]:
import os

print("Fichiers KPI créés :")

for file in os.listdir(PROCESSED_PATH):
    if "kpi" in file.lower():
        print("-", file)

Fichiers KPI créés :
- daily_kpis.csv
- exercise_kpis.csv
- game_kpis.csv
- global_kpis.csv
- level_kpis.csv
- patient_dashboard_kpis.csv
- patient_kpis.csv


: 

# Conclusion

Le Notebook 05 a permis de transformer les données analytiques en
indicateurs de performance utilisables pour le suivi de la rééducation.

Les KPI ont été calculés à plusieurs niveaux :

- global ;
- patient ;
- exercice ;
- jeu ;
- niveau ;
- évolution temporelle.

Les principaux indicateurs calculés sont :

- nombre de patients ;
- nombre de sessions ;
- score moyen ;
- taux de réussite moyen ;
- durée moyenne ;
- nombre moyen de répétitions ;
- progression moyenne ;
- nombre de sessions avec amélioration ;
- nombre de sessions avec diminution.

Un tableau spécifique `patient_dashboard_kpis` a également été créé
afin de regrouper les principaux indicateurs nécessaires au suivi
individuel des patients.

Les résultats ont été sauvegardés dans le dossier `data/processed/`
afin d'être réutilisés dans les prochaines étapes du projet.

Le prochain notebook permettra d'approfondir l'analyse de la progression
des patients au cours du temps.